In [ ]:
# CELL 1: GOOGLE COLAB SETUP & DRIVE MOUNT (OPTIONAL)
from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
# CELL 2: CONFIGURATION & FILE PATHS
# Provide your Google Colab CSV paths below:
TRAIN_DATASET_PATH = "/content/PASTE_TRAIN_DATASET_HERE.csv"

# List of multiple test files to evaluate:
TEST_DATASET_PATHS = [
    "/content/PASTE_TEST_DATASET_1_HERE.csv",
    "/content/PASTE_TEST_DATASET_2_HERE.csv"
]

MODEL_SAVE_PATH = "/content/mlp_trained_model.npz"


In [ ]:
# CELL 3: IMPORTS & HELPER FUNCTIONS
import numpy as np
import matplotlib.pyplot as plt
import os

def load_csv_data(file_path):
    """
    Reads a CSV file manually, handles comma/tab delimiters,
    skips headers, and separates numerical features from labels.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}. Please check your path.")

    features = []
    labels = []

    with open(file_path, "r") as f:
        lines = f.readlines()

    # Determine delimiter and check for header
    first_line = lines[0].strip()
    delimiter = "\t" if "\t" in first_line else ","
    start_idx = 0

    first_tokens = first_line.split(delimiter)
    try:
        float(first_tokens[0])
    except ValueError:
        start_idx = 1  # Skip header row

    for line in lines[start_idx:]:
        line = line.strip()
        if not line:
            continue
        parts = line.split(delimiter)
        # All columns except last are features
        row_feat = [float(val.strip()) for val in parts[:-1]]
        # Last column is the class label
        row_label = parts[-1].strip()

        features.append(row_feat)
        labels.append(row_label)

    return np.array(features, dtype=np.float64), labels

def one_hot_encode(labels, unique_classes):
    """Encodes string/categorical labels into one-hot binary vectors."""
    one_hot = np.zeros((len(labels), len(unique_classes)), dtype=np.float64)
    for i, label in enumerate(labels):
        col_idx = unique_classes.index(label)
        one_hot[i, col_idx] = 1.0
    return one_hot

def sigmoid(z):
    """Sigmoid activation function with clipping to avoid overflow."""
    z_clipped = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z_clipped))

def sigmoid_derivative(a):
    """Derivative of sigmoid given output activation a."""
    return a * (1.0 - a)

def softmax(z):
    """Softmax activation for multi-class probability outputs."""
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)


In [ ]:
# CELL 4: MODULE 1 - TRAINING MODULE
def train_mlp():
    print("=" * 60)
    print("MODULE 1: TRAINING MULTI-LAYER PERCEPTRON")
    print("=" * 60)

    # 1. Read Training Dataset
    print(f"Reading training data from: {TRAIN_DATASET_PATH}")
    X_train, y_train_raw = load_csv_data(TRAIN_DATASET_PATH)

    unique_classes = sorted(list(set(y_train_raw)))
    y_train = one_hot_encode(y_train_raw, unique_classes)

    num_samples, num_features = X_train.shape
    num_classes = len(unique_classes)
    print(f"Training Samples: {num_samples} | Features: {num_features} | Classes: {num_classes} ({unique_classes})")

    # 2. Min-Max Normalization
    feat_min = np.min(X_train, axis=0)
    feat_max = np.max(X_train, axis=0)
    feat_diff = np.where((feat_max - feat_min) == 0, 1.0, feat_max - feat_min)
    X_train_norm = (X_train - feat_min) / feat_diff

    # 3. Model Architecture & Initialization
    num_hidden = 8
    learning_rate = 0.05
    epochs = 300

    np.random.seed(42)
    W1 = np.random.uniform(-0.5, 0.5, (num_features, num_hidden))
    b1 = np.zeros((1, num_hidden))
    W2 = np.random.uniform(-0.5, 0.5, (num_hidden, num_classes))
    b2 = np.zeros((1, num_classes))

    loss_history = []
    accuracy_history = []

    # 4. Training Loop
    for epoch in range(epochs):
        # Forward pass
        z1 = np.dot(X_train_norm, W1) + b1
        a1 = sigmoid(z1)
        z2 = np.dot(a1, W2) + b2
        y_pred_prob = softmax(z2)

        # Categorical cross-entropy loss
        loss = -np.mean(np.sum(y_train * np.log(y_pred_prob + 1e-12), axis=1))
        loss_history.append(loss)

        # Accuracy
        pred_labels = np.argmax(y_pred_prob, axis=1)
        actual_labels = np.argmax(y_train, axis=1)
        acc = np.mean(pred_labels == actual_labels) * 100.0
        accuracy_history.append(acc)

        # Backpropagation
        delta2 = (y_pred_prob - y_train) / num_samples
        dW2 = np.dot(a1.T, delta2)
        db2 = np.sum(delta2, axis=0, keepdims=True)

        delta1 = np.dot(delta2, W2.T) * sigmoid_derivative(a1)
        dW1 = np.dot(X_train_norm.T, delta1)
        db1 = np.sum(delta1, axis=0, keepdims=True)

        # Gradient descent update
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2
        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1

        if (epoch + 1) % 50 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d}/{epochs} | Loss: {loss:.4f} | Training Accuracy: {acc:.2f}%")

    print(f"\nFinal Training Accuracy: {accuracy_history[-1]:.2f}%")

    # 5. Save Model Parameters to File
    np.savez(
        MODEL_SAVE_PATH,
        W1=W1, b1=b1, W2=W2, b2=b2,
        feat_min=feat_min, feat_diff=feat_diff,
        classes=np.array(unique_classes)
    )
    print(f"Model parameters successfully saved to: {MODEL_SAVE_PATH}")

    # 6. Plot Training Curves
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(range(1, epochs + 1), loss_history, color="red")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("MLP: Epoch vs Training Loss")
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(range(1, epochs + 1), accuracy_history, color="blue")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.title("MLP: Epoch vs Training Accuracy")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# Run Training
train_mlp()


In [ ]:
# CELL 5: MODULE 2 - TESTING MODULE (NO RETRAINING)
def test_mlp():
    print("=" * 60)
    print("MODULE 2: TESTING MULTI-LAYER PERCEPTRON (NO RETRAINING)")
    print("=" * 60)

    # 1. Load Trained Parameters
    if not os.path.exists(MODEL_SAVE_PATH):
        raise FileNotFoundError(f"Model file {MODEL_SAVE_PATH} not found! Please run training cell first.")

    model_data = np.load(MODEL_SAVE_PATH)
    W1 = model_data["W1"]
    b1 = model_data["b1"]
    W2 = model_data["W2"]
    b2 = model_data["b2"]
    feat_min = model_data["feat_min"]
    feat_diff = model_data["feat_diff"]
    unique_classes = list(model_data["classes"])

    print(f"Loaded trained model from: {MODEL_SAVE_PATH}")
    print(f"Recognized Classes: {unique_classes}")

    # 2. Iterate Over Multiple Test Files
    for file_idx, test_path in enumerate(TEST_DATASET_PATHS, start=1):
        print("\n" + "-" * 50)
        print(f"Evaluating Test File {file_idx}: {test_path}")
        print("-" * 50)

        if not os.path.exists(test_path):
            print(f"Warning: File {test_path} not found. Skipping.")
            continue

        X_test, y_test_raw = load_csv_data(test_path)

        # 3. Normalize using Saved Training Parameters
        X_test_norm = (X_test - feat_min) / feat_diff

        # 4. Forward Pass Only
        z1 = np.dot(X_test_norm, W1) + b1
        a1 = sigmoid(z1)
        z2 = np.dot(a1, W2) + b2
        test_probs = softmax(z2)

        pred_indices = np.argmax(test_probs, axis=1)
        predicted_classes = [unique_classes[idx] for idx in pred_indices]

        # 5. Evaluate Accuracy
        correct = 0
        for i in range(len(y_test_raw)):
            if predicted_classes[i] == y_test_raw[i]:
                correct += 1

        accuracy = (correct / len(y_test_raw)) * 100.0
        print(f"Total Test Samples: {len(y_test_raw)}")
        print(f"Correct Predictions: {correct}/{len(y_test_raw)}")
        print(f"Final Test Accuracy: {accuracy:.2f}%")

        # Display Sample Predictions
        print("\nSample Predictions (First 5 samples):")
        print(f"{'Actual':<20} | {'Predicted':<20} | Status")
        print("-" * 50)
        for i in range(min(5, len(y_test_raw))):
            status = "MATCH" if y_test_raw[i] == predicted_classes[i] else "MISMATCH"
            print(f"{y_test_raw[i]:<20} | {predicted_classes[i]:<20} | {status}")

# Run Testing
test_mlp()


## 3. Important Viva Questions & Answers
1. **Q: Why can an MLP solve the XOR problem while an SLP cannot?**  
   *Ans:* SLP can only create a single linear hyperplane. XOR is not linearly separable. MLP uses hidden layers with non-linear activation functions to transform the feature space into a higher dimension where classes become linearly separable.
2. **Q: What is the role of the activation function?**  
   *Ans:* Non-linear activation functions prevent multi-layer combinations from collapsing into a single equivalent linear transformation ($W_2(W_1 X) = W' X$).
3. **Q: Why must test data be normalized using the training min/max values?**  
   *Ans:* Test data must be projected into the exact same scaled space learned by the model during training. Using test set min/max would cause distribution shift and data leakage.
4. **Q: What is the vanishing gradient problem?**  
   *Ans:* For functions like Sigmoid whose maximum derivative is $0.25$, multiplying gradients backwards across multiple layers shrinks gradients exponentially, causing early layers to train very slowly.
